# Agent 5 — Video Editing Agent (VEA)

> Compose a long-form video into a short-form output: highlight reel,
> recap, trailer-style cut. Async via `/video/edit` — the final asset
> arrives at your configured webhook URL.

## Why this one is async

VEA jobs run for minutes on long inputs. Holding an HTTP connection
open that long is fragile, so the endpoint is fire-and-forget:

1. You `POST /video/edit` with a list of asset_ids + a natural-language
   edit prompt. The endpoint returns immediately with a `task_id`.
2. When the edit finishes (or fails), Memories.ai POSTs a callback to
   the webhook URL you configured at
   [api-platform.memories.ai/webhooks](https://api-platform.memories.ai/webhooks).
3. Your webhook handler picks up the callback and (optionally)
   downloads the result asset.

> **Important**: `/video/edit` returns
> `400 "async request requires at least one webhook"` if no webhook
> is configured. Configure one before running this notebook end-to-end.

## Endpoints exercised

| Step | Endpoint |
|---|---|
| Upload to VI | `POST /upload` (Asset Management) |
| Wait for indexing | `GET /{asset_id}/metadata` (poll) |
| Optional scene detection | `POST /video/clip` |
| Edit | `POST /video/edit` |


## Setup

You need:

1. A Memories.ai API key (`sk-mavi-...`) — get one at the [Memories.ai console](https://api-platform.memories.ai/stripe).
2. Python 3.10+ and the `requests` library (`pip install requests`).

Set the key as an environment variable before launching Jupyter, or paste it
inline in the cell below. The same key works across Visual Search, Visual
Intelligence, and Visual Agents — no separate auth per product.


In [ ]:
import os, json, time, requests

# ────────────────────────────────────────────────────────────────────────────
# Auth: the Memories.ai key is a single token used across every product.
# Pass it as the literal `Authorization` header value — no `Bearer` prefix.
# ────────────────────────────────────────────────────────────────────────────
API_KEY = os.environ.get("MEMORIES_API_KEY") or "sk-mavi-..."  # ← paste here if not using env
HEADERS = {"Authorization": API_KEY}

# Hosts: Visual Search and Visual Intelligence live on different domains.
VS_HOST  = "https://api.memories.ai/serve/api/v1"            # Visual Search
VLM_HOST = "https://mavi-backend.memories.ai/serve/api/v2"   # Visual Intelligence (VLM + Visual Agents)

# Default VLM model used for verification / reasoning. Other options include
# `qwen:qwen2.5-vl-72b-instruct`, `nova:amazon.nova-lite-v1:0`, etc. — see
# Memories.ai docs for the full list and per-token pricing.
VLM_MODEL = "gemini:gemini-2.5-flash"


## Helper — VI Asset Management

VEA operates on `re_xxx` assets (Visual Intelligence Asset Management),
which is a *different* upload pipeline than Visual Search's `VI_xxx`
videos. The endpoints live on the VI host (`mavi-backend`), not the
Visual Search host (`api.memories.ai`).


In [ ]:
def upload_asset(file_path):
    """VI Asset Management — POST /upload. Returns {asset_id: 're_xxx'}.

    The asset will be in WAITING/UPLOADING state initially and must
    reach SUCCESS before it can be used by /video/edit.
    """
    with open(file_path, "rb") as f:
        r = requests.post(
            f"{VLM_HOST}/upload",
            headers=HEADERS,
            files={"file": (os.path.basename(file_path), f, "video/mp4")},
            timeout=300,
        )
    r.raise_for_status()
    envelope = r.json()
    assert envelope.get("code") in (200, "200"), envelope
    return envelope["data"]


def wait_for_asset_ready(asset_id, *, timeout_sec=600):
    """Poll GET /{asset_id}/metadata until upload_status == SUCCESS."""
    deadline = time.time() + timeout_sec
    delay = 5.0
    last = None
    while time.time() < deadline:
        r = requests.get(f"{VLM_HOST}/{asset_id}/metadata",
                         headers=HEADERS, timeout=30)
        r.raise_for_status()
        body = r.json()
        resource = (body.get("data") or {}).get("resource") or [{}]
        status = resource[0].get("upload_status")
        if status != last:
            print(f"  upload_status -> {status!r}")
            last = status
        if status == "SUCCESS":
            return resource[0]
        if status == "FAILED":
            raise RuntimeError(f"upload failed: {body}")
        time.sleep(delay)
        delay = min(delay * 1.5, 20.0)
    raise TimeoutError(f"asset {asset_id} did not reach SUCCESS in {timeout_sec}s")


## Step 1 — get a `re_xxx` asset

You need a Visual Intelligence asset_id (`re_xxx`) to edit. Three paths:

- **Paste an existing asset_id** you already have in your account.
- **Auto-fetch the public test asset** (what this notebook does by default
  on Run-All).
- **Upload your own local file** — see the helper functions above.


In [ ]:
import urllib.request

# If you already have an asset, set it here and skip the auto-upload below.
ASSET_ID = ""

if not ASSET_ID:
    # Run-All default: download the public test video, upload it to VI,
    # wait for it to reach SUCCESS, then proceed.
    PUBLIC_VIDEO = "https://storage.googleapis.com/memories-test-data/test_1min.mp4"
    local_path = "/tmp/vea_test_video.mp4"
    if not os.path.exists(local_path):
        print(f"Downloading {PUBLIC_VIDEO} ...")
        urllib.request.urlretrieve(PUBLIC_VIDEO, local_path)
    print("Uploading to VI Asset Management ...")
    asset_meta = upload_asset(local_path)
    ASSET_ID = asset_meta["asset_id"]
    print(f"Got ASSET_ID = {ASSET_ID}")
    wait_for_asset_ready(ASSET_ID)

print(f"Editing with asset {ASSET_ID}")


## Step 2 — *(optional)* scene detection

`/video/clip` is automatic scene-boundary detection. It's async; the
boundaries arrive via the same webhook. Skip if your prompt doesn't
depend on scene breaks.


In [ ]:
def video_clip(asset_id):
    """POST /video/clip — async scene detection. Returns {task_id}."""
    r = requests.post(f"{VLM_HOST}/video/clip", headers=HEADERS,
                      json={"asset_id": asset_id}, timeout=60)
    r.raise_for_status()
    envelope = r.json()
    assert envelope.get("code") in (200, "200"), envelope
    return envelope["data"]

# Uncomment to fire it:
# clip_task = video_clip(ASSET_ID)
# print("scene-detection task_id:", clip_task["task_id"])


## Step 3 — submit the edit

The `user_prompt` is plain English — keep it under 500 characters and
prefer English for best results. The model decides the output duration
based on your prompt and the source.


In [ ]:
def video_edit(asset_ids, user_prompt, *, orientation="landscape"):
    """POST /video/edit — async AI-driven edit. Returns {task_id}.

    Raises if no webhook is configured on the account
    ("async request requires at least one webhook").
    """
    r = requests.post(
        f"{VLM_HOST}/video/edit",
        headers=HEADERS,
        json={
            "asset_ids": list(asset_ids),
            "orientation": orientation,
            "user_prompt": user_prompt,
        },
        timeout=60,
    )
    r.raise_for_status()
    envelope = r.json()
    if envelope.get("code") not in (200, "200"):
        raise RuntimeError(f"/video/edit failed: {envelope}")
    return envelope["data"]


edit_task = video_edit(
    [ASSET_ID],
    "Pick the most visually interesting 30 seconds of this video for a landscape highlight clip.",
    orientation="landscape",
)
print(f"edit task_id: {edit_task['task_id']}")
print("\nThe final asset will arrive at your configured webhook with this task_id.")


## Step 4 — receiving the callback (out of scope here)

Memories.ai will POST a JSON body to your webhook when the edit
completes. Shape:

```json
{
  "code": 200,
  "message": "SUCCESS",
  "data": {
    "asset_id": "re_<the rendered output>",
    "extension": "mp4",
    "file_size": 62217261,
    "upload_status": "SUCCESS"
  },
  "task_id": "<matches edit_task['task_id']>"
}
```

Match the `task_id` to correlate this callback with your `edit_task`,
then download the rendered file via the VI download endpoint or hand
it to your video pipeline.

For local testing without writing a server, [webhook.site](https://webhook.site)
gives you a free public URL that captures incoming requests. Configure
it at [api-platform.memories.ai/webhooks](https://api-platform.memories.ai/webhooks).
